# PV-gradient and eddy-tilt direction case studies

This notebook automatically finds clear examples of the polarity-dependent relationship between eddy tilt and the ambient shallow-water PV gradient. The working hypotheses are:

- In the planetary-beta-dominant open ocean, cyclonic eddies (CEs) preferentially tilt **along** the PV gradient, whereas anticyclonic eddies (AEs) preferentially tilt **opposite** it.
- On strong topographic gradients, CEs continue to align as the total PV gradient rotates toward the topographic contribution.
- Topographic-dominant AEs show weaker or inconsistent directional preference.

Candidates are selected automatically, but the final figures must still be checked for coherent tracks, data coverage, coastline effects, and physically meaningful transitions.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

HERE = Path.cwd().resolve()
ANALYSIS_ROOT = next(
    (p for p in (HERE, *HERE.parents) if (p / 'seacofs_tilt_tools.py').exists()),
    None,
)
if ANALYSIS_ROOT is None:
    raise FileNotFoundError('Run from seacofs_eddy_tilt_analysis or one of its subfolders.')
CASE_ROOT = ANALYSIS_ROOT / 'case_studies'
for path in (CASE_ROOT, ANALYSIS_ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import seacofs_tilt_tools as tilt
from case_study_tools import (
    PVAlignmentConfig,
    plot_pv_alignment_case,
    rank_pv_alignment_cases,
    select_pv_alignment_cases,
)

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 50)

## 1. Load PV-gradient diagnostics

The primary screen uses core-mean bathymetry and bathymetric gradients. Direction is considered reliable only when tilt distance is at least 5 km; very small displacement vectors have unstable bearings and should not drive case selection.

In [ ]:
paths = tilt.Paths()
grid = tilt.load_grid(paths.grid, paths.z_r)
df_eddies, _ = tilt.load_tilt_tables(paths)
df_eddies = tilt.add_region_labels(df_eddies, grid)
df_eddies = tilt.add_pv_gradient_terms(df_eddies, grid, core_mean=True)

required = {
    'Eddy', 'Day', 'Cyc', 'h', 'TiltDis', 'TiltDir',
    'PV_grad_theta', 'PV_grad_mag', 'PV_grad_plan_mag',
    'PV_grad_topo_mag', 'dtheta_PV_grad', 'topo_plan_ratio',
}
missing = required - set(df_eddies.columns)
if missing:
    raise KeyError(f'Missing required columns: {sorted(missing)}')
if df_eddies.duplicated(['Eddy', 'Day']).any():
    raise ValueError('Expected one row per Eddy-Day.')

print(f"{len(df_eddies):,} observations from {df_eddies.Eddy.nunique():,} eddies")
display(df_eddies.groupby('Cyc').agg(observations=('Eddy', 'size'), eddies=('Eddy', 'nunique')))

## 2. Selection criteria

Planetary or topographic dominance requires the relevant PV-gradient magnitude to be at least twice the competing contribution. Open-ocean candidates must additionally have core-mean depth of at least 2,000 m. Alignment and opposition allow a ±45° tolerance.

The ranking rewards sustained behaviour, not isolated matching observations. It also rewards long lifetime, time spent in the target regime, directional-data coverage, and small typical target error. AE and CE open-ocean cases are ranked separately.

In [ ]:
config = PVAlignmentConfig(
    smooth_window=7,
    min_periods=5,
    min_lifetime_days=60,
    min_valid_observations=20,
    min_regime_observations=10,
    min_sustained_run=5,
    dominance_factor=2.0,
    angle_tolerance_deg=45.0,
    min_open_ocean_depth_m=2000.0,
    min_tilt_distance_km=5.0,
)

df_direction, ranking = rank_pv_alignment_cases(df_eddies, config)
display(
    ranking.groupby('ranking_group')['eligible']
    .agg(candidates='size', eligible='sum')
)

## 3. Ranked case-study candidates

The four groups are: open-ocean AE opposition, open-ocean CE alignment, topographic CE alignment, and topographic AE directional dispersion. For the AE topographic group, lower circular resultant length means that relative directions are more dispersed and therefore show less directional preference.

In [ ]:
ranking_columns = [
    'Eddy', 'Cyc', 'Region', 'case_score', 'lifetime_days',
    'regime_observations', 'regime_fraction', 'longest_regime_run',
    'matching_fraction', 'longest_matching_run',
    'median_target_error_deg', 'relative_angle_resultant',
    'alignment_fraction', 'opposition_fraction',
    'directional_coverage', 'median_tilt_km', 'median_depth_m',
]
group_titles = {
    'open_ocean_ce': 'Open-ocean CE alignment',
    'open_ocean_ae': 'Open-ocean AE opposition',
    'topographic_ce_alignment': 'Topographic CE alignment',
    'topographic_ae_no_preference': 'Topographic AE weak preference',
}
for group, label in group_titles.items():
    print(f'\n{label}')
    display(
        ranking.loc[(ranking.ranking_group == group) & ranking.eligible, ranking_columns]
        .head(20).round(3)
)

## 4. Population context before selecting individual eddies

These distributions show whether the selected cases sit within the broader polarity-dependent signal. They help prevent the case studies from being interpreted as the only behaviour in the dataset.

In [ ]:
planetary = df_direction[df_direction.open_ocean_planetary & df_direction.direction_valid]
topographic = df_direction[df_direction.topographic_strong & df_direction.direction_valid]
bins = np.arange(0, 181, 10)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for cyc, color in [('AE', 'tab:red'), ('CE', 'tab:blue')]:
    axes[0].hist(planetary.loc[planetary.Cyc == cyc, 'dtheta_PV_grad'], bins=bins,
                 density=True, histtype='step', lw=2, color=color, label=cyc)
    axes[1].hist(topographic.loc[topographic.Cyc == cyc, 'dtheta_PV_grad'], bins=bins,
                 density=True, histtype='step', lw=2, color=color, label=cyc)
for ax, title in zip(axes, ['Planetary ≥ 2× and depth ≥ 2,000 m', 'Topographic ≥ 2×']):
    ax.axvspan(0, 45, color='tab:blue', alpha=.08)
    ax.axvspan(135, 180, color='tab:orange', alpha=.08)
    ax.set(title=title, xlabel='|Tilt − PV gradient| (°)', ylabel='Density')
    ax.legend(frameon=False)
fig.suptitle('Population context for directional case studies');

## 5. Select equally sized candidate sets

The default is three eddies per group. The same eddy may be scientifically useful in more than one group if it spends different parts of its lifetime offshore and over a slope. Keep such overlap when it illustrates a within-eddy transition; otherwise replace it with the next ranked candidate.

In [ ]:
N_PER_GROUP = 3
selected = select_pv_alignment_cases(ranking, n_per_group=N_PER_GROUP)
selected

In [ ]:
selected_long = pd.DataFrame(
    [(group, eddy) for group, eddies in selected.items() for eddy in eddies],
    columns=['case_group', 'Eddy'],
)
display(selected_long.groupby('Eddy').filter(lambda x: len(x) > 1).sort_values('Eddy'))

## 6. Direction and bathymetry figures

The first panel shows tilt and total PV-gradient compass bearings without connecting across the artificial 0°/360° boundary. For AEs, green markers show the direction opposite the PV gradient. The second panel shows `dtheta_PV_grad`: 0° is alignment and 180° is opposition. Remaining panels show dominance, component magnitudes, tilt magnitude/depth, and the full track over local bathymetry.

In [ ]:
for group, eddies in selected.items():
    print(f'\n{group_titles[group]}')
    for eddy_id in eddies:
        track = df_direction[df_direction.Eddy == eddy_id].copy()
        fig, _, _ = plot_pv_alignment_case(
            track, grid, config=config,
            title=f'{group_titles[group]} — {track.Cyc.iloc[0]} eddy {eddy_id}'
)
        plt.show()

## 7. Regime-specific summaries for selected cases

These summaries quantify the directional behaviour visible in each figure. Fractions are calculated only from observations with tilt distance ≥5 km. Day-level values are descriptive repeated observations, not statistically independent samples.

In [ ]:
selected_ids = sorted(set(selected_long.Eddy))
case_rows = []
for eddy_id in selected_ids:
    part = df_direction[df_direction.Eddy == eddy_id]
    for regime_name, mask in {
        'open_ocean_planetary': part.open_ocean_planetary,
        'topographic': part.topographic_strong,
    }.items():
        use = part[mask & part.direction_valid]
        if use.empty:
            continue
        case_rows.append({
            'Eddy': eddy_id, 'Cyc': part.Cyc.iloc[0], 'regime': regime_name,
            'observations': len(use),
            'median_dtheta_deg': use.dtheta_PV_grad.median(),
            'aligned_within_45_fraction': (use.dtheta_PV_grad <= 45).mean(),
            'opposed_within_45_fraction': (use.dtheta_PV_grad >= 135).mean(),
            'median_tilt_km': use.TiltDis.median(),
            'median_depth_m': use.h.median(),
        })
case_summary = pd.DataFrame(case_rows)
display(case_summary.round(3).sort_values(['Cyc', 'Eddy', 'regime']))

## 8. Dominance-threshold sensitivity

A case is stronger if it remains highly ranked when dominance is tightened from 2:1 to 4:1. Failure at 4:1 does not invalidate a 2:1 case, but it indicates that its directional behaviour occurs where planetary and topographic contributions are less cleanly separated.

In [ ]:
strict_config = PVAlignmentConfig(
    smooth_window=config.smooth_window, min_periods=config.min_periods,
    min_lifetime_days=config.min_lifetime_days,
    min_valid_observations=config.min_valid_observations,
    min_regime_observations=config.min_regime_observations,
    min_sustained_run=config.min_sustained_run, dominance_factor=4.0,
    angle_tolerance_deg=config.angle_tolerance_deg,
    min_open_ocean_depth_m=config.min_open_ocean_depth_m,
    min_tilt_distance_km=config.min_tilt_distance_km,
)
_, strict_ranking = rank_pv_alignment_cases(df_eddies, strict_config)
strict_eligible = set(
    zip(
        strict_ranking.loc[strict_ranking.eligible, 'ranking_group'],
        strict_ranking.loc[strict_ranking.eligible, 'Eddy'],
    )
)
sensitivity = selected_long.copy()
sensitivity['eligible_at_4_to_1'] = [
    (row.case_group, row.Eddy) in strict_eligible for row in sensitivity.itertuples()
]
display(sensitivity)

## Interpretation checklist

A strong case should have: (1) sustained rather than isolated alignment/opposition; (2) a clear dominance regime; (3) adequate tilt magnitude and directional coverage; (4) a coherent track over the expected bathymetric setting; (5) simultaneous motion of `TiltDir`, `PV_grad_theta`, and `dtheta_PV_grad`; and (6) no obvious boundary or missing-data artefact.

The figures demonstrate representative evolution, not causality by themselves. The population distributions and threshold sensitivity should accompany any highlighted case so the examples remain tied to the complete eddy census.